In [6]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


# Export calibrated parameters

Exports every parameter defined in `parameters_projections_jablje.py` and `parameters_projections_rakican.py` (calibrated crop parameters, soil layers, groundwater, field management, initial conditions, ...) to one long-format CSV file per site (`parameter`, `value` columns). A metadata header with authorship/contact/version info is prepended as commented lines in each file.

In [7]:
import dataclasses
from datetime import date
from pathlib import Path

import pandas as pd

from aquacrop_slovenia import parameters_projections_jablje as jablje_module
from aquacrop_slovenia import parameters_projections_rakican as rakican_module
from aquacrop_slovenia import config

## Flatten parameter modules into (parameter, value) pairs

Every module-level variable prefixed with the site name (`jablje_...` / `rakican_...`) is picked up automatically, so adding a new variable to the parameter files is enough to have it included in the export - no need to touch this notebook. Dataclasses (e.g. `SoilLayer`), lists (e.g. multiple soil layers) and objects with a `.params` dict (e.g. `FieldManagement`, `InitialConditions`, `GroundWater`) are recursively flattened into dotted / indexed parameter names.

In [8]:
def flatten(prefix: str, value, rows: list[tuple[str, object]]) -> None:
    """Recursively flatten dataclasses / objects / dicts / lists into (parameter, value) rows."""
    if dataclasses.is_dataclass(value) and not isinstance(value, type):
        flatten(prefix, dataclasses.asdict(value), rows)
    elif isinstance(value, dict):
        for key, val in value.items():
            flatten(f"{prefix}.{key}", val, rows)
    elif isinstance(value, (list, tuple)):
        for i, val in enumerate(value):
            flatten(f"{prefix}[{i}]", val, rows)
    elif hasattr(value, "__dict__"):
        flatten(prefix, vars(value), rows)
    else:
        rows.append((prefix, value))


def module_to_rows(site: str, module) -> list[tuple[str, object]]:
    """Flatten every `<site>_...` variable defined in a parameters module."""
    rows: list[tuple[str, object]] = []
    for var_name, value in vars(module).items():
        if not var_name.startswith(f"{site}_"):
            continue
        param_name = var_name[len(site) + 1 :]  # strip the "<site>_" prefix
        flatten(param_name, value, rows)
    return rows

In [9]:
sites = {
    "jablje": jablje_module,
    "rakican": rakican_module,
}

params_by_site = {
    site: pd.DataFrame.from_records(
        [{"parameter": parameter, "value": value} for parameter, value in module_to_rows(site, module)],
        columns=["parameter", "value"],
    )
    for site, module in sites.items()
}

params_by_site["jablje"]

,parameter,value
0,maize_params.crop_type,2
1,maize_params.is_sown,True
2,maize_params.cycle_determination,0
3,maize_params.adjust_for_eto,True
4,maize_params.base_temp,10.0
...,...,...
141,initial_cond.params.water_layer_ec,0.0
142,initial_cond.params.soil_water_content_type,0
143,initial_cond.params.soil_data[0].thickness,4.0
144,initial_cond.params.soil_data[0].water_content,36.3


## Metadata header

Add/edit any additional info lines here before exporting.

In [16]:
source_files = {
    "jablje": "aquacrop_slovenia/parameters_projections_jablje.py",
    "rakican": "aquacrop_slovenia/parameters_projections_rakican.py",
}


def build_header_lines(site: str, param_version: str) -> list[str]:
    header_info = {
        "Title": f"Calibrated AquaCrop parameters - {site.capitalize()} maize site, Slovenia",
        "Authors": "Rok Kuk, Ajda Bleiweis, Tanja Travnikar, Lazar Pavić, Aleš Kolmanič, Nejc Golob, Petra Pantar, Tjaša Pogačar",
        "Contact": "rok.kuk@bf.uni-lj.si",
        "Date exported": date.today().isoformat(),
        "Parameter version": param_version,
        "Aquacrop version": config.AQUACROP_VERSION,
        "Source file": source_files[site],
        "License": "CC BY 4.0",
        "DOI": "10.5281/zenodo.21829886"
    }
    return [f"# {key}: {value}" for key, value in header_info.items()]


print("\n".join(build_header_lines("jablje", "1.0")))

# Title: Calibrated AquaCrop parameters - Jablje maize site, Slovenia
# Authors: Rok Kuk
# Contact: rok.kuk@bf.uni-lj.si
# Date exported: 2026-08-19
# Parameter version: 1.0
# Aquacrop version: 7.1
# Source file: aquacrop_slovenia/parameters_projections_jablje.py
# License: CC BY 4.0
# DOI: 10.5281/zenodo.21829886


## Export to CSV

In [ ]:
param_version = "1.0" # Change when parameters change

In [17]:
output_dir = Path("../../results")
output_dir.mkdir(parents=True, exist_ok=True)

for site, df in params_by_site.items():
    output_path = output_dir / f"calibrated_parameters_{site}_v{param_version}.csv"
    with output_path.open("w", encoding="utf-8", newline="") as f:
        f.write("\n".join(build_header_lines(site, param_version)) + "\n")
        df.to_csv(f, index=False, lineterminator="\n")
    print(f"Wrote {len(df)} parameter rows to {output_path.resolve()}")

Wrote 146 parameter rows to C:\Users\rokku\Documents\Code\aquacrop-slovenia\results\calibrated_parameters_jablje_v1.0.csv
Wrote 282 parameter rows to C:\Users\rokku\Documents\Code\aquacrop-slovenia\results\calibrated_parameters_rakican_v1.0.csv


## Generate pyaquacrop input files

Run the calibrated model once per site on the historical station data purely to have `pyaquacrop` generate the AquaCrop input files it derives from these parameters (climate, crop, soil, management, initial conditions, project file, ...). The run itself and its outputs are discarded afterwards; only the generated input folders (`DATA`, `LIST`, `OBS`, `PARAM`, `SIMUL`) are kept, under `results/aquacrop_files_<site>/`.

In [18]:
param_version = "1.0" # Change when parameters change

In [27]:
import shutil

from aquacrop_slovenia.yield_projections import run_historical_simulation

# Subdirectories pyaquacrop populates with the *input* files it generates from the
# parameters. Everything else it creates (OUTP, results, the copied executable, ...)
# is run output/tooling and gets discarded.
INPUT_DIRS = {"DATA", "LIST", "SIMUL"}

for site in sites:
    site_dir = output_dir / f"aquacrop_files_calibrated_parameters_{site}_v{param_version}"
    shutil.rmtree(site_dir, ignore_errors=True)

    run_historical_simulation(site, working_dir=site_dir, cleanup=False)

    for entry in site_dir.iterdir():
        if entry.is_dir() and entry.name in INPUT_DIRS:
            continue
        #shutil.rmtree(entry) if entry.is_dir() else entry.unlink()

    print(f"Kept pyaquacrop input files for {site} in {site_dir.resolve()}")

Setting up working directory at: C:\Users\rokku\Documents\Code\aquacrop-slovenia\results\aquacrop_files_calibrated_parameters_jablje_v1.0
Running AquaCrop simulation with project file: C:\Users\rokku\Documents\Code\aquacrop-slovenia\results\aquacrop_files_calibrated_parameters_jablje_v1.0\LIST\PROJECT.PRM
Detected platform: windows (AMD64)
Looking for executable at: C:\Users\rokku\Documents\Code\aquacrop-slovenia\.venv\Lib\site-packages\model\windows\aquacrop.exe
Using AquaCrop executable: C:\Users\rokku\Documents\Code\aquacrop-slovenia\results\aquacrop_files_calibrated_parameters_jablje_v1.0\aquacrop.exe
AquaCrop simulation completed successfully
Parsing simulation results...
Results successfully parsed
Kept pyaquacrop input files for jablje in C:\Users\rokku\Documents\Code\aquacrop-slovenia\results\aquacrop_files_calibrated_parameters_jablje_v1.0
Setting up working directory at: C:\Users\rokku\Documents\Code\aquacrop-slovenia\results\aquacrop_files_calibrated_parameters_rakican_v1.0
